<a href="https://colab.research.google.com/github/adeleke1508/flyrank_week_one_ml_project_1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adeleke1508/flyrank_week_one_ml_project_1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

In [3]:
df = pd.read_csv('/content/drive/MyDrive/dim_content_and_fact_daily_performance_month_2026_05_cleaned.csv')
df.columns

Index(['client_hash_id_x', 'content_hash_id', 'keyword_hash_id', 'url_hash_id',
       'keyword_char_count', 'keyword_token_count', 'url_char_count',
       'content_created_date', 'content_updated_date', 'content_type',
       'search_volume', 'competition', 'competition_level', 'cpc',
       'main_intent', 'backlinks', 'category_count', 'keyword_created_date',
       'provider_used', 'model_used', 'char_count', 'word_count',
       'last_optimized_date', 'optimization_eligible_date', 'is_published',
       'is_deleted', 'report_date', 'client_hash_id_y', 'client_has_gsc',
       'client_has_ga4', 'gsc_data_available', 'ga4_data_available',
       'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position',
       'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions',
       'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct',
       'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
       'ai_chatgpt', 'ai_perplexity', 'a

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293868 entries, 0 to 293867
Data columns (total 59 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   client_hash_id_x            293868 non-null  object 
 1   content_hash_id             293868 non-null  object 
 2   keyword_hash_id             293868 non-null  object 
 3   url_hash_id                 293868 non-null  object 
 4   keyword_char_count          293868 non-null  int64  
 5   keyword_token_count         293868 non-null  int64  
 6   url_char_count              293868 non-null  int64  
 7   content_created_date        293868 non-null  object 
 8   content_updated_date        293868 non-null  object 
 9   content_type                293868 non-null  object 
 10  search_volume               293868 non-null  float64
 11  competition                 293868 non-null  float64
 12  competition_level           293868 non-null  object 
 13  cpc           

In [5]:
date_cols = ['content_created_date', 'content_updated_date']
df[date_cols] = df[date_cols].apply(pd.to_datetime, errors='coerce')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293868 entries, 0 to 293867
Data columns (total 59 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   client_hash_id_x            293868 non-null  object        
 1   content_hash_id             293868 non-null  object        
 2   keyword_hash_id             293868 non-null  object        
 3   url_hash_id                 293868 non-null  object        
 4   keyword_char_count          293868 non-null  int64         
 5   keyword_token_count         293868 non-null  int64         
 6   url_char_count              293868 non-null  int64         
 7   content_created_date        293868 non-null  datetime64[ns]
 8   content_updated_date        293868 non-null  datetime64[ns]
 9   content_type                293868 non-null  object        
 10  search_volume               293868 non-null  float64       
 11  competition                 293868 non-

In [6]:
df['ctr_fixed'] = df['ctr'] * 100
#I am doing this because the ctr was gotten from ; ctr = clicks/((impressions) * 100), hence giving the data too much margin.

In [7]:
df['content_created_date'].max()

Timestamp('2026-05-31 00:00:00')

In [8]:
df['content_created_date'].min()

Timestamp('2026-01-20 00:00:00')

In [9]:
earliest_date_of_creation = df['content_created_date'].min()
latest_date_of_creation = df['content_created_date'].max()

earliest_date_of_update = df['content_updated_date'].min()
latest_date_of_upate = df['content_updated_date'].max()

In [10]:
df['old'] = np.where(df['content_created_date'] <= '2026-03-15' , 1, 0)
conditions = (df['old']==1) & (df['content_updated_date'] <='2026-06-13')
df['old_but_updated_recently'] = np.where(conditions, 0, 1)

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method:

I use KMeans on nine structural features (word_count, char_count, keyword_token_count, url_char_count, category_count, backlinks, search_volume, competition, cpc; log-transformed and standardised on train). Performance columns are excluded so archetypes are not built from the outcome. content_type (2 of 6,819 training pages differ) and main_intent (rare categories) were left out after tiny categories formed their own clusters under scaling. Because clusters have no ground truth, I test usefulness against my Week-4 baseline: does expected CTR by position × archetype predict held-out CTR better than position alone, using impression-weighted MAE on the same test pages? A shallow decision tree is a supervised reference and permutation importance shows what it used. I did not use Random Forest or Gradient Boosting: they add complexity without answering whether archetypes carry signal. k=3 was chosen on structure-only criteria (no tiny clusters, stable across seeds) before any CTR results were seen.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: I split at the page level by content_created_date: older pages train, newer pages test, with a check that no content_hash_id appears on both sides. This matches the real use of the benchmark, which will be applied to content newer than the content it learned from, and it guarantees the model never sees a test page during training. All learned parts (scaler, clusters, benchmarks) are fit on the training pages only.

Limits: the outcome metrics come from the same report window for both groups, so this is an older-vs-newer split, not a forecast over time, and age differences between the groups may contribute to any gap. Clients are not held out, because [reason, or “I also report a client-held-out version”]; one client contributes most of the top-ranked pages, so I report errors per client to check that the result isn’t driven by it.

In [12]:
df['content_created_date'].describe()

,content_created_date
count,293868
mean,2026-03-01 09:56:45.953693184
min,2026-01-20 00:00:00
25%,2026-02-05 00:00:00
50%,2026-02-25 00:00:00
75%,2026-03-21 00:00:00
max,2026-05-31 00:00:00


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

d = df.copy()
d['report_date'] = pd.to_datetime(d['report_date'])
d['pos_x_imp'] = d['gsc_avg_position'] * d['gsc_impressions']

In [14]:
struct_cols = ['client_hash_id_x', 'content_type', 'main_intent', 'word_count', 'char_count',
               'keyword_token_count', 'url_char_count', 'category_count',
               'backlinks', 'search_volume', 'competition', 'cpc']
check = d.groupby('content_hash_id')[['content_created_date'] + struct_cols].nunique()
print("pages with >1 value per column (should all be 0):")
print((check > 1).sum())

pages with >1 value per column (should all be 0):
content_created_date    0
client_hash_id_x        0
content_type            0
main_intent             0
word_count              0
char_count              0
keyword_token_count     0
url_char_count          0
category_count          0
backlinks               0
search_volume           0
competition             0
cpc                     0
dtype: int64


In [15]:
pages = (d.sort_values('report_date')).groupby('content_hash_id').agg(client=('client_hash_id_x', 'first'),
                content_type=('content_type', 'first'),
                main_intent=('main_intent', 'first'),
                created=('content_created_date', 'min'),
                word_count=('word_count', 'first'),
                char_count=('char_count', 'first'),
                keyword_token_count=('keyword_token_count', 'first'),
                url_char_count=('url_char_count', 'first'),
                category_count=('category_count', 'first'),
                backlinks=('backlinks', 'first'),
                search_volume=('search_volume', 'first'),
                competition=('competition', 'first'),
                cpc=('cpc', 'first'),
                impressions=('gsc_impressions', 'sum'),
                clicks=('gsc_clicks', 'sum'),
                old=('old', 'last'),
                old_but_updated_recently=('old_but_updated_recently', 'last'),
                pos_x_imp=('pos_x_imp', 'sum'),
                n_reports=('report_date', 'nunique')).reset_index()

In [16]:
pages['avg_position'] = pages['pos_x_imp'] / pages['impressions']
pages['ctr'] = pages['clicks'] / pages['impressions']
pages = pages[pages['impressions'] >= 100].copy()
pages['pos_bucket'] = pd.cut(pages['avg_position'], [0, 3, 5, 7, 10, 20, 50, 100])

print("pages:", len(pages), "| unique ids:", pages['content_hash_id'].nunique())

pages: 9467 | unique ids: 9467


In [17]:
print(pages['created'].describe())
for q in [0.6, 0.7, 0.8]:
    print(f"{q:.0%} of pages were created before", pages['created'].quantile(q).date())

count                             9467
mean     2026-03-04 04:54:01.406992896
min                2026-01-20 00:00:00
25%                2026-02-05 00:00:00
50%                2026-02-25 00:00:00
75%                2026-03-26 00:00:00
max                2026-05-31 00:00:00
Name: created, dtype: object
60% of pages were created before 2026-03-06
70% of pages were created before 2026-03-17
80% of pages were created before 2026-04-08


In [18]:
CUTOFF = pd.Timestamp('2026-03-21')
train = pages[pages['created'] < CUTOFF].copy()
test  = pages[pages['created'] >= CUTOFF].copy()

In [19]:
overlap = set(train['content_hash_id']) & set(test['content_hash_id'])
print("\npage overlap between train and test:", len(overlap))
assert len(overlap) == 0


page overlap between train and test: 0


In [20]:
print("pages      train/test:", len(train), len(test))
print("impressions train/test:", train['impressions'].sum(), test['impressions'].sum())
print("test share of pages: %.1f%%" % (100 * len(test) / len(pages)))

pages      train/test: 6819 2648
impressions train/test: 33008832 7125689
test share of pages: 28.0%


In [21]:
print("\nclients in train:", train['client'].nunique(), "| in test:", test['client'].nunique())
print("clients appearing in both:", len(set(train['client']) & set(test['client'])))
print("\ntest pages per client (top 5):")
print(test['client'].value_counts().head())


clients in train: 11 | in test: 18
clients appearing in both: 11

test pages per client (top 5):
client
client_06d356715a8ff3b6    632
client_86ebc2f12c01f586    565
client_1a730cb2640a1abf    386
client_73cda7b4e4f265ea    168
client_e5c2aa26a8598242    139
Name: count, dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [22]:
pages.head(10)

,content_hash_id,client,content_type,main_intent,created,word_count,char_count,keyword_token_count,url_char_count,category_count,...,cpc,impressions,clicks,old,old_but_updated_recently,pos_x_imp,n_reports,avg_position,ctr,pos_bucket
1,content_000c32172de71cf0,client_73cda7b4e4f265ea,keyword article,transactional,2026-02-10,2569.0,16152.0,9,126,4,...,0.14,10544,18,1,1,73671.0,31,6.987007,0.001707,"(5, 7]"
2,content_0010f59b0934069a,client_e5c2aa26a8598242,keyword article,transactional,2026-02-05,2827.0,17205.0,7,105,7,...,0.00,4267,13,1,0,34054.0,31,7.980783,0.003047,"(7, 10]"
3,content_001d139d4c8bee7b,client_86ebc2f12c01f586,keyword article,informational,2026-04-21,2932.0,18631.0,8,103,0,...,0.00,1163,14,0,1,20528.0,26,17.650903,0.012038,"(10, 20]"
4,content_00275616dbe76014,client_fef1a8f436438636,keyword article,commercial,2026-01-29,3238.0,19359.0,9,119,6,...,0.00,3055,3,1,1,19097.0,31,6.251064,0.000982,"(5, 7]"
5,content_002f5941c8b369e0,client_73cda7b4e4f265ea,keyword article,commercial,2026-02-17,2702.0,17863.0,9,97,4,...,0.04,7177,24,1,1,51126.0,31,7.123589,0.003344,"(7, 10]"
6,content_003a723a2958becd,client_86ebc2f12c01f586,keyword article,informational,2026-04-30,2733.0,18337.0,5,113,3,...,0.00,2617,5,0,1,56115.0,31,21.442491,0.001911,"(20, 50]"
7,content_00483aa0af320d5b,client_e5c2aa26a8598242,keyword article,informational,2026-02-25,2405.0,15136.0,9,107,7,...,0.00,1489,9,1,1,37901.0,31,25.453996,0.006044,"(20, 50]"
8,content_004bc79184bd186f,client_1a730cb2640a1abf,keyword article,informational,2026-04-24,1800.0,12004.0,3,97,6,...,0.00,285,3,0,1,1661.0,5,5.828070,0.010526,"(5, 7]"
9,content_0054d0c689d4bcc1,client_86ebc2f12c01f586,keyword article,informational,2026-04-20,3073.0,19687.0,6,96,0,...,0.00,2468,9,0,1,23363.0,31,9.466370,0.003647,"(7, 10]"
10,content_00598d80aca0da65,client_86ebc2f12c01f586,keyword article,informational,2026-04-14,2965.0,19121.0,7,97,0,...,0.00,22488,118,0,1,132548.0,31,5.894166,0.005247,"(5, 7]"


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
NUM = ['word_count','char_count','keyword_token_count','url_char_count',
       'category_count','backlinks','search_volume','competition','cpc']

In [24]:
def make_X(df):
  X = np.log1p(df[NUM].clip(lower=0))
  return X

In [25]:
train_x = make_X(train)
test_x = make_X(test).reindex(columns=train_x.columns, fill_value=0.0)

In [26]:
sc = StandardScaler().fit(train_x)
scaled_x_train , scaled_x_test = sc.transform(train_x), sc.transform(test_x)

In [27]:
rng = np.random.RandomState(0)
sub = rng.choice(len(scaled_x_train), size=min(8000, len(scaled_x_train)), replace=False)
rows = []
for k in range(3, 9):
  labs = [KMeans(k, n_init=5, random_state=s).fit(scaled_x_train).labels_ for s in  (1, 2, 3)]
  rows.append(dict(k=k,
                     silhouette=silhouette_score(scaled_x_train[sub], labs[0][sub]),
                     stability_ari=np.mean([adjusted_rand_score(labs[0], labs[j]) for j in (1, 2)]),
                     smallest_cluster=np.bincount(labs[0]).min()))
print(pd.DataFrame(rows).round(3))

   k  silhouette  stability_ari  smallest_cluster
0  3       0.283          0.999               973
1  4       0.286          0.891                20
2  5       0.246          0.764                20
3  6       0.256          0.999                20
4  7       0.258          1.000                20
5  8       0.222          0.845                20


In [28]:
K = 3
km = KMeans(K, n_init=10, random_state=42).fit(scaled_x_train)
train['cluster'] = km.labels_
test['cluster']  = km.predict(scaled_x_test)

In [29]:
prof = train.groupby('cluster').agg(
    pages=('ctr','size'), words=('word_count','median'), backlinks=('backlinks','median'),
    volume=('search_volume','median'), cpc=('cpc','median'),
    top_intent=('main_intent', lambda s: s.mode().iat[0]),
    avg_pos=('avg_position','median'),
    impressions=('impressions','median'),
    weighted_ctr=('clicks', 'sum'))
prof['weighted_ctr'] = prof['weighted_ctr'] / train.groupby('cluster')['impressions'].sum()
print(prof.round(4))

         pages   words  backlinks  volume   cpc     top_intent  avg_pos  \
cluster                                                                   
0         3075  2724.0        0.0     0.0  0.00  informational  11.2215   
1          979  2625.0       48.0    30.0  1.48  transactional  11.3766   
2         2765  2586.0        1.0    20.0  0.00  informational   9.7834   

         impressions  weighted_ctr  
cluster                             
0             2286.0        0.0042  
1             1256.0        0.0037  
2             1778.0        0.0039  


In [30]:
train.head()

,content_hash_id,client,content_type,main_intent,created,word_count,char_count,keyword_token_count,url_char_count,category_count,...,impressions,clicks,old,old_but_updated_recently,pos_x_imp,n_reports,avg_position,ctr,pos_bucket,cluster
1,content_000c32172de71cf0,client_73cda7b4e4f265ea,keyword article,transactional,2026-02-10,2569.0,16152.0,9,126,4,...,10544,18,1,1,73671.0,31,6.987007,0.001707,"(5, 7]",2
2,content_0010f59b0934069a,client_e5c2aa26a8598242,keyword article,transactional,2026-02-05,2827.0,17205.0,7,105,7,...,4267,13,1,0,34054.0,31,7.980783,0.003047,"(7, 10]",2
4,content_00275616dbe76014,client_fef1a8f436438636,keyword article,commercial,2026-01-29,3238.0,19359.0,9,119,6,...,3055,3,1,1,19097.0,31,6.251064,0.000982,"(5, 7]",2
5,content_002f5941c8b369e0,client_73cda7b4e4f265ea,keyword article,commercial,2026-02-17,2702.0,17863.0,9,97,4,...,7177,24,1,1,51126.0,31,7.123589,0.003344,"(7, 10]",2
7,content_00483aa0af320d5b,client_e5c2aa26a8598242,keyword article,informational,2026-02-25,2405.0,15136.0,9,107,7,...,1489,9,1,1,37901.0,31,25.453996,0.006044,"(20, 50]",2


In [31]:
prof

,pages,words,backlinks,volume,cpc,top_intent,avg_pos,impressions,weighted_ctr
cluster,,,,,,,,,
0,3075,2724.0,0.0,0.0,0.00,informational,11.221521,2286.0,0.004165
1,979,2625.0,48.0,30.0,1.48,transactional,11.376634,1256.0,0.003711
2,2765,2586.0,1.0,20.0,0.00,informational,9.783439,1778.0,0.003903


In [32]:
for df in (train, test):
  df['pb'] = df['pos_bucket'].astype(str)

In [33]:
#position (Week 4 Baseline Rule)
g = train.groupby('pb')[['clicks', 'impressions']].sum()
mapA = g['clicks']/g['impressions']
test['predA'] = test['pb'].map(mapA)

In [34]:
#position x archetype, with fallback to A when a cell is too small
MIN_N = 50
cell = train.groupby(['pb', 'cluster']).agg(n=('ctr', 'size'), clicks=('clicks', 'sum'), imps=('impressions', 'sum'))
cell['pred'] = cell['clicks'] / cell['imps']
cell = cell.loc[cell['n'] >= MIN_N, 'pred']
test = test.join(cell.rename('predB'), on=['pb','cluster'])
print("share of test pages using the fallback in B: %.1f%%" % (100 * test['predB'].isna().mean()))
test['predB'] = test['predB'].fillna(test['predA'])

share of test pages using the fallback in B: 1.6%


In [35]:
type(scaled_x_train)

numpy.ndarray

In [36]:
Xtr = pd.DataFrame(scaled_x_train, columns=train_x.columns)
Xte = pd.DataFrame(scaled_x_test, columns=train_x.columns)

In [37]:
#shallow decision tree on position + structure (no cluster label)
from sklearn.tree import DecisionTreeRegressor
XtrC = Xtr.assign(avg_position=train['avg_position'])
XteC = Xte.assign(avg_position=test['avg_position'])
tree = DecisionTreeRegressor(max_depth=4, min_samples_leaf=100, random_state=0)
tree.fit(XtrC, train['ctr'], sample_weight=train['impressions'])
test['predC'] = tree.predict(XteC)

In [38]:
def wmae(y, p, w): return np.average(np.abs(y - p), weights=w)

y, w = test['ctr'].values, test['impressions'].values
res = pd.DataFrame({m: [wmae(y, test['pred'+m].values, w)] for m in 'ABC'}, index=['weighted_MAE']).T
res['vs_A_%'] = 100 * (res.loc['A','weighted_MAE'] - res['weighted_MAE']) / res.loc['A','weighted_MAE']
print(res.round(6))

def boot_diff(pa, px, n=1000, seed=0):
    r = np.random.default_rng(seed); ea, ex = np.abs(y-pa), np.abs(y-px); out = []
    for _ in range(n):
        s = r.integers(0, len(y), len(y))
        out.append(np.average(ex[s]-ea[s], weights=w[s]))
    return np.percentile(out, [2.5, 50, 97.5])

for m in 'BC':
    lo, mid, hi = boot_diff(test['predA'].values, test['pred'+m].values)
    print(f"{m} minus A: median {mid:.6f}, 95% interval [{lo:.6f}, {hi:.6f}]")

   weighted_MAE    vs_A_%
A      0.002710  0.000000
B      0.002719 -0.331309
C      0.002967 -9.489216
B minus A: median 0.000010, 95% interval [-0.000020, 0.000036]
C minus A: median 0.000260, 95% interval [0.000167, 0.000347]


In [39]:
# ranking check: same eligibility filter as the Week-4 rule
elig = (test['old'] == 1) & (test['old_but_updated_recently'] == 0)
top = {}
for m in 'ABC':
    test['miss'+m] = (test['impressions'] * (test['pred'+m] - test['ctr'])).clip(lower=0)
    top[m] = set(test[elig].nlargest(20, 'miss'+m)['content_hash_id'])
print("top-20 overlap  A&B:", len(top['A'] & top['B']), " A&C:", len(top['A'] & top['C']), " B&C:", len(top['B'] & top['C']))

top-20 overlap  A&B: 0  A&C: 0  B&C: 0


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

What the errors look like. Errors are similar across the three structural clusters (weighted MAE 0.0026 to 0.0028), and the archetype-aware benchmark is within about 2% of the position-only baseline in every cluster (0.7% and 2.3% worse in clusters 0 and 1, 0.4% better in cluster 2). Differences between clients are much larger: weighted MAE runs from 0.0017 to 0.0035 across the five largest test clients, while the A-vs-B gap within any client is at most 0.0001. This suggests that CTR differs by client in ways neither position nor page structure captures, and that archetype carries little additional information about CTR in this data.

In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# where does each benchmark go wrong?
def slice_err(col):
    return test.groupby(col).apply(lambda d: pd.Series({
        'pages': len(d), 'MAE_A': wmae(d['ctr'], d['predA'], d['impressions']),
        'MAE_B': wmae(d['ctr'], d['predB'], d['impressions'])}))
print(slice_err('cluster').round(5))
print(slice_err('client').sort_values('pages', ascending=False).head(5).round(5))

          pages    MAE_A    MAE_B
cluster                          
0        1361.0  0.00281  0.00283
1         258.0  0.00264  0.00270
2        1029.0  0.00258  0.00257
                         pages    MAE_A    MAE_B
client                                          
client_06d356715a8ff3b6  632.0  0.00323  0.00324
client_86ebc2f12c01f586  565.0  0.00241  0.00242
client_1a730cb2640a1abf  386.0  0.00349  0.00359
client_73cda7b4e4f265ea  168.0  0.00169  0.00171
client_e5c2aa26a8598242  139.0  0.00232  0.00225


/tmp/ipykernel_100778/2062954912.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return test.groupby(col).apply(lambda d: pd.Series({
/tmp/ipykernel_100778/2062954912.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return test.groupby(col).apply(lambda d: pd.Series({


In [41]:
y, w = test['ctr'].values, test['impressions'].values
test['predK'] = train['clicks'].sum() / train['impressions'].sum()      # constant benchmark
def wrmse(y, p, w): return np.sqrt(np.average((y - p) ** 2, weights=w))
summary = pd.DataFrame({m: dict(MAE=wmae(y, test['pred'+m], w), RMSE=wrmse(y, test['pred'+m], w),
                                bias=np.average(test['pred'+m] - y, weights=w)) for m in 'KABC'}).T
summary.index = ['constant', 'A position', 'B position x archetype', 'C tree']
print(summary.round(6))

floor = np.average(np.sqrt(test['predA'] * (1 - test['predA']) / test['impressions']) * np.sqrt(2 / np.pi), weights=w)
print('approx. noise floor (MAE): %.6f' % floor)

test['zero_clicks'] = test['clicks'] == 0
for col in ['pb', 'zero_clicks']:
    print(test.groupby(col).apply(lambda d: pd.Series({'pages': len(d),
          'MAE_A': wmae(d['ctr'], d['predA'], d['impressions']),
          'MAE_B': wmae(d['ctr'], d['predB'], d['impressions'])}), include_groups=False).round(5))

# ranking: the Week-4 filter matches no test page, so compare on all test pages
print('eligible test pages:', elig.sum())
tops = {m: set(test.nlargest(20, 'miss'+m)['content_hash_id']) for m in 'ABC'}
print('overlap A&B:', len(tops['A'] & tops['B']), ' A&C:', len(tops['A'] & tops['C']), ' B&C:', len(tops['B'] & tops['C']))

from sklearn.inspection import permutation_importance
pi = permutation_importance(tree, XteC, test['ctr'], sample_weight=test['impressions'],
                            scoring='neg_mean_absolute_error', n_repeats=10, random_state=0)
print(pd.Series(pi.importances_mean, index=XteC.columns).sort_values(ascending=False).round(6))

                             MAE      RMSE      bias
constant                0.002883  0.004502 -0.000885
A position              0.002710  0.004350 -0.000937
B position x archetype  0.002719  0.004345 -0.000853
C tree                  0.002967  0.004575 -0.000804
approx. noise floor (MAE): 0.000776
            pages    MAE_A    MAE_B
pb                                 
(0, 3]       19.0  0.00412  0.00412
(10, 20]   1108.0  0.00260  0.00270
(20, 50]    357.0  0.00231  0.00230
(3, 5]      166.0  0.00429  0.00420
(5, 7]      422.0  0.00273  0.00272
(50, 100]    10.0  0.00090  0.00090
(7, 10]     566.0  0.00207  0.00206
              pages    MAE_A    MAE_B
zero_clicks                          
False        2241.0  0.00268  0.00269
True          407.0  0.00391  0.00392
eligible test pages: 0
overlap A&B: 18  A&C: 13  B&C: 12
word_count             0.000093
avg_position           0.000006
backlinks              0.000000
keyword_token_count    0.000000
category_count         0.000000
url_ch

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.